# 🎯 TÁI LẬP STAIR BASELINE TRÊN TẬP DỮ LIỆU TIKTOK (TRI-MODAL MICRO-VIDEO)
## 🏆 Mô hình STAIR Gốc (AAAI 2025) | Embedding Dim = 64D | Batch Size = 1024 | Tri-modal (Vision + Text + Audio) | Epochs = 500 | Checkpoint: NDCG@20
*(Tái tạo chuẩn tắc mốc đối chuẩn nền tảng STAIR Baseline trên tập video ngắn đa giác quan: **TikTok (DiffMM / ACM MM 2024 - 9,308 Users, 6,710 Items)**)*
---
### 📌 CẤU HÌNH THAM SỐ CHUẨN TẮC (PAPER CONFIG):
* **Dataset:** `tiktok` (Tương tác: 59,541 train / 3,051 valid / 6,130 test).
* **Đa phương thức (Tri-modal):** `visual_modality.pkl` (128D), `textual_modality.pkl` (768D), `audio_modality.pkl` (128D).
* **Đồ thị k-NN Tam phân:** `num_neighbors: '3-3-3'` (phân bổ đều $3$ láng giềng cho mỗi phương thức, tỷ lệ $33.3\%$ mỗi kênh).
* **Suy giảm phổ FSC:** $\gamma = 0.05$ (bảo tồn thông tin đa phương tiện dải tần trung và cao trong video ngắn).
* **Embedding Dim:** `64D` | **Số tầng tích chập:** `L = 3` | **Batch Size:** `1024`.
* **Bộ tối ưu:** `AdamWSEvo` ($lr = 10^{-3}$, weight decay $= 0.1$, $\beta_1 = 0.9, \beta_2 = 0.999$).
* **Checkpoint Selection:** Giám sát `NDCG@20` cao nhất trên tập Validation (`which4best: NDCG@20`), sau đó đánh giá chính thức trên tập TEST.


## Cell 1 ⚙️ Thiết lập Môi trường, Dependencies & Đồng bộ STAIR-Enhanced
Khởi tạo môi trường Kaggle, clone/đồng bộ mã nguồn mới nhất từ branch `main` của repository [STAIR-Enhanced](https://github.com/HenryBui777/STAIR-Enhanced.git), cài đặt các thư viện bắt buộc (`freerec==0.8.5`, `torchdata==0.7.1`, `torch_geometric`, `pynvml`, `prettytable`, `scipy`), và nhúng mã nguồn `main.py` cùng tệp cấu hình `tiktok_MMRec.yaml`.

In [ ]:
# Cell 1: Môi trường, Dependencies & Đồng bộ STAIR-Enhanced
import os, shutil, subprocess, sys

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
os.chdir('/kaggle/working')

# 1. Đồng bộ repository mới nhất từ origin/main
if os.path.exists(STAIR_DIR):
    print("Thư mục STAIR-Enhanced đã tồn tại. Đang đồng bộ cưỡng bức mã nguồn mới nhất...")
    try:
        subprocess.run(['git', '-C', STAIR_DIR, 'fetch', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', STAIR_DIR, 'reset', '--hard', 'origin/main'], check=True)
        print("✅ Đã reset về commit mới nhất của origin/main.")
    except Exception as e:
        print(f"Lỗi git fetch/reset ({e}), đang làm sạch và clone lại từ đầu...")
        shutil.rmtree(STAIR_DIR, ignore_errors=True)

if not os.path.exists(STAIR_DIR):
    print("Cloning STAIR-Enhanced repository (branch main)..." )
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/HenryBui777/STAIR-Enhanced.git', STAIR_DIR
    ], check=True)

for p in [STAIR_DIR, '/kaggle/working']:
    if p not in sys.path:
        sys.path.insert(0, p)

if os.path.exists(STAIR_DIR):
    os.chdir(STAIR_DIR)

# Ghi đảm bảo main.py và configs/tiktok_MMRec.yaml tồn tại trên đĩa
os.makedirs(os.path.join(STAIR_DIR, 'configs'), exist_ok=True)
with open(os.path.join(STAIR_DIR, 'main.py'), 'w', encoding='utf-8') as f:
    f.write('\n\nfrom typing import Dict, Tuple, Optional\n\nimport torch, os, math\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport freerec\n\nfrom optimizers.Adam import AdamSEvo\nfrom optimizers.AdamW import AdamWSEvo\nfrom optimizers.utils import Smoother\n\nfreerec.declare(version=\'1.0.1\')\n\ncfg = freerec.parser.Parser()\ncfg.add_argument("--embedding-dim", type=int, default=64)\ncfg.add_argument("--num-layers", type=int, default=3, help="the number of layers for FSC/BSC")\n\ncfg.add_argument("--mfiles", type=str, default="textual_modality.pkl,visual_modality.pkl", help="the files saving modality")\ncfg.add_argument("--num-neighbors", type=str, default=\'5-1\', help="for kNN graph")\ncfg.add_argument("--gamma", type=float, default=0.2)\n\ncfg.set_defaults(\n    description="STAIR",\n    root="../../data",\n    dataset=\'Amazon2014Baby_550_MMRec\',\n    epochs=500,\n    batch_size=1024,\n    optimizer=\'adamwsevo\',\n    lr=1e-3,\n    weight_decay=0.1,\n    seed=1,\n    monitors=["Recall@10", "Recall@20", "NDCG@10", "NDCG@20"],\n    which4best="NDCG@20",\n)\ncfg.compile()\n\n\nif isinstance(cfg.mfiles, str):\n    cfg.mfiles = cfg.mfiles.split(\',\')\nif isinstance(cfg.num_neighbors, str):\n    cfg.num_neighbors = list(map(int, cfg.num_neighbors.split(\'-\'))) \n\n# beta3 here is the 1 - beta_j for BSC\ncfg.beta3 = (0.1 + 0.9 * (torch.arange(cfg.embedding_dim) / cfg.embedding_dim).pow(cfg.gamma)).to(cfg.device)\n\nclass STAIR(freerec.models.GenRecArch):\n\n    def __init__(\n        self, dataset: freerec.data.datasets.RecDataSet\n    ) -> None:\n        super().__init__(dataset)\n\n        self.num_layers = cfg.num_layers\n\n        self.User.add_module(\n            "embeddings", nn.Embedding(\n                self.User.count, cfg.embedding_dim\n            )\n        )\n\n        self.Item.add_module(\n            "embeddings", nn.Embedding(\n                self.Item.count, cfg.embedding_dim\n            )\n        )\n\n        self.register_buffer(\n            "Adj",\n            self.dataset.train().to_normalized_adj(\n                normalization=\'sym\'\n            )\n        )\n\n        self.reset_parameters()\n\n        self.prepare(dataset.path)\n\n        self.criterion = freerec.criterions.BPRLoss(reduction=\'mean\')\n\n    def reset_parameters(self):\n        for m in self.modules():\n            if isinstance(m, nn.Linear):\n                nn.init.kaiming_normal_(m.weight)\n                if m.bias is not None:\n                    nn.init.constant_(m.bias, 0.)\n            elif isinstance(m, nn.Embedding):\n                nn.init.normal_(m.weight, std=1.e-4)\n            elif isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):\n                nn.init.constant_(m.weight, 1.)\n                nn.init.constant_(m.bias, 0.)\n\n    def marked_params(self):\n        params = [\n            {\n                \'params\': self.User.parameters(),\n                \'smoother\': None\n            },\n            {\n                \'params\': self.Item.parameters(), \n                \'smoother\': Smoother(self.mAdj, beta=cfg.beta3, L=cfg.num_layers, aggr=\'neumann\')\n            },\n        ]\n        return params\n\n    def whitening(self, feats: torch.Tensor):\n        if not isinstance(feats, torch.Tensor):\n            feats = torch.tensor(feats, dtype=torch.float32)\n        else:\n            feats = feats.float()\n        feats = feats - feats.mean(0, keepdim=True)\n        feats, _, _ = torch.linalg.svd(feats, full_matrices=False)\n        if feats.size(1) < cfg.embedding_dim:\n            reps = math.ceil(cfg.embedding_dim / feats.size(1))\n            scale = math.sqrt(feats.size(1) / cfg.embedding_dim)\n            feats = (feats.repeat(1, reps)[:, :cfg.embedding_dim]) * scale\n        else:\n            feats = feats[:, :cfg.embedding_dim]\n        return feats * math.sqrt(self.Item.count / cfg.embedding_dim)\n\n    def get_knn_graph(self, features: torch.Tensor, k: int = 5):\n        r"""\n        Compute the kNN graph.\n        """\n        if not isinstance(features, torch.Tensor):\n            features = torch.tensor(features, dtype=torch.float32)\n        else:\n            features = features.float()\n        features = F.normalize(features, dim=-1) # (N, D)\n        sim = features @ features.t() # (N, N)\n        sim.fill_diagonal_(-10.)\n        edge_index, _ = freerec.graph.get_knn_graph(\n            sim, k, symmetric=False\n        )\n        return edge_index\n\n    def prepare(self, path: str):\n        from freerec.utils import import_pickle\n\n        mfeats = []\n        for mfile in cfg.mfiles:\n            mpath = os.path.join(path, mfile)\n            if not os.path.exists(mpath):\n                # Fallback search across common data locations\n                for cand in [\n                    os.path.join(cfg.root, cfg.dataset, mfile),\n                    os.path.join("/kaggle/data", cfg.dataset, mfile),\n                    os.path.join("/kaggle/working/STAIR/data", cfg.dataset, mfile),\n                    os.path.join("/kaggle/working/STAIR-Enhanced/data", cfg.dataset, mfile),\n                    os.path.join("data", cfg.dataset, mfile),\n                ]:\n                    if os.path.exists(cand):\n                        mpath = cand\n                        break\n            mfeats.append(import_pickle(mpath))\n\n        edge_index = torch.cat(\n            [self.get_knn_graph(feats, k) for feats, k in zip(mfeats, cfg.num_neighbors)],\n            dim=1\n        )\n        edge_weight = torch.ones_like(edge_index[0], dtype=torch.float)\n        edge_index, edge_weight = freerec.graph.coalesce(\n            edge_index, edge_weight, reduce=\'sum\'\n        )\n        edge_index, edge_weight = freerec.graph.to_undirected(\n            edge_index, edge_weight, reduce=\'max\'\n        )\n        edge_index, edge_weight = freerec.graph.to_normalized(\n            edge_index, edge_weight,\n            normalization=\'sym\'\n        )\n        mAdj = torch.sparse_coo_tensor(\n            edge_index, edge_weight,\n            size=(self.Item.count, self.Item.count)\n        )\n        self.register_buffer(\n            \'mAdj\',\n            mAdj.to_sparse_csr()\n        )\n\n        # MI\n        mfeats = [self.whitening(mfeat) * k for mfeat, k in zip(mfeats, cfg.num_neighbors)]\n        mfeats = sum(mfeats).div(sum(cfg.num_neighbors))\n        self.Item.embeddings.weight.data.copy_(mfeats)\n\n        edge_index = self.dataset.train().to_bigraph(edge_type=\'u2i\')[\'u2i\'].edge_index\n        edge_index, edge_weight = freerec.graph.to_normalized(edge_index, normalization=\'left\')\n        R = torch.sparse_coo_tensor(\n            edge_index, edge_weight, size=(self.User.count, self.Item.count)\n        ).to_sparse_csr()\n\n        self.User.embeddings.weight.data.copy_(R @ mfeats)\n\n    def sure_trainpipe(self, batch_size: int):\n        return self.dataset.train().shuffled_pairs_source(\n        ).gen_train_sampling_neg_(\n            num_negatives=1\n        ).batch_(batch_size).tensor_()\n\n    def encode(self) -> Tuple[torch.Tensor, torch.Tensor]:\n        allEmbds = torch.cat(\n            (self.User.embeddings.weight, self.Item.embeddings.weight), dim=0\n        ) # (N, D)\n\n        features = allEmbds\n        smoothed = allEmbds\n        \n        # FSC\n        beta = 1 - cfg.beta3\n        norm_correction = 1 - beta ** (self.num_layers + 1)\n        for _ in range(self.num_layers):\n            features = self.Adj @ features * beta\n            smoothed = smoothed + features\n        avgEmbds = smoothed.mul(1 - beta).div(norm_correction)\n        userEmbds, itemEmbds = torch.split(\n            avgEmbds, (self.User.count, self.Item.count)\n        )\n        return userEmbds, itemEmbds\n\n    def fit(self, data: Dict[freerec.data.fields.Field, torch.Tensor]):\n        userEmbds, itemEmbds = self.encode()\n        users, positives, negatives = data[self.User], data[self.Item], data[self.INeg]\n        userEmbds = userEmbds[users] # (B, 1, D)\n        iposEmbds = itemEmbds[positives] # (B, 1, D)\n        inegEmbds = itemEmbds[negatives] # (B, K, D)\n\n        rec_loss = self.criterion(\n            torch.einsum("BKD,BKD->BK", userEmbds, iposEmbds),\n            torch.einsum("BKD,BKD->BK", userEmbds, inegEmbds)\n        )\n        return rec_loss\n\n    def reset_ranking_buffers(self):\n        """This method will be executed before evaluation."""\n        userEmbds, itemEmbds = self.encode()\n        self.ranking_buffer = dict()\n        self.ranking_buffer[self.User] = userEmbds.detach().clone()\n        self.ranking_buffer[self.Item] = itemEmbds.detach().clone()\n\n    def recommend_from_full(self, data: Dict[freerec.data.fields.Field, torch.Tensor]):\n        userEmbds = self.ranking_buffer[self.User][data[self.User]] # (B, 1, D)\n        itemEmbds = self.ranking_buffer[self.Item]\n        return torch.einsum("BKD,ND->BN", userEmbds, itemEmbds)\n\n    def recommend_from_pool(self, data: Dict[freerec.data.fields.Field, torch.Tensor]):\n        userEmbds = self.ranking_buffer[self.User][data[self.User]] # (B, 1, D)\n        itemEmbds = self.ranking_buffer[self.Item][data[self.IUnseen]] # (B, 101, D)\n        return torch.einsum("BKD,BKD->BK", userEmbds, itemEmbds)\n\n\nclass CoachForSTAIR(freerec.launcher.Coach):\n\n    def set_optimizer(self):\n        if self.cfg.optimizer.lower() == \'sgd\':\n            self.optimizer = torch.optim.SGD(\n                self.model.marked_params(), lr=self.cfg.lr, \n                momentum=self.cfg.momentum,\n                nesterov=self.cfg.nesterov,\n                weight_decay=self.cfg.weight_decay\n            )\n        elif self.cfg.optimizer.lower() == \'adam\':\n            self.optimizer = torch.optim.Adam(\n                self.model.marked_params(), lr=self.cfg.lr,\n                betas=(self.cfg.beta1, self.cfg.beta2),\n                weight_decay=self.cfg.weight_decay\n            )\n        elif self.cfg.optimizer.lower() == \'adamw\':\n            self.optimizer = torch.optim.AdamW(\n                self.model.marked_params(), lr=self.cfg.lr,\n                betas=(self.cfg.beta1, self.cfg.beta2),\n                weight_decay=self.cfg.weight_decay\n            )\n        elif self.cfg.optimizer.lower() == \'adamsevo\':\n            self.optimizer = AdamSEvo(\n                self.model.marked_params(), lr=self.cfg.lr,\n                betas=(self.cfg.beta1, self.cfg.beta2),\n                weight_decay=self.cfg.weight_decay\n            )\n        elif self.cfg.optimizer.lower() == \'adamwsevo\':\n            self.optimizer = AdamWSEvo(\n                self.model.marked_params(), lr=self.cfg.lr,\n                betas=(self.cfg.beta1, self.cfg.beta2),\n                weight_decay=self.cfg.weight_decay\n            )\n        else:\n            raise NotImplementedError(\n                f"Unexpected optimizer {self.cfg.optimizer} ..."\n            )\n\n    def train_per_epoch(self, epoch: int):\n        for data in self.dataloader:\n            data = self.dict_to_device(data)\n            loss = self.model(data)\n\n            self.optimizer.zero_grad()\n            loss.backward()\n            self.optimizer.step()\n            \n            self.monitor(\n                loss.item(), \n                n=len(data[self.User]), reduction="mean", \n                mode=\'train\', pool=[\'LOSS\']\n            )\n\n\ndef main():\n\n    # Robust auto-bridge for FreeRec:\n    processed_dir = os.path.join(cfg.root, "Processed", cfg.dataset)\n    if os.path.islink(processed_dir) and not os.path.exists(processed_dir):\n        try:\n            os.unlink(processed_dir)\n        except Exception:\n            pass\n\n    if not os.path.exists(processed_dir) or (os.path.isdir(processed_dir) and not os.listdir(processed_dir)):\n        script_dir = os.path.dirname(os.path.abspath(__file__)) if \'__file__\' in locals() else \'.\'\n        candidates = [\n            os.path.join(cfg.root, cfg.dataset),\n            os.path.join("/kaggle/data", cfg.dataset),\n            os.path.join("/kaggle/data/Processed", cfg.dataset),\n            os.path.join("/kaggle/working/STAIR/data", cfg.dataset),\n            os.path.join("/kaggle/working/STAIR-Enhanced/data", cfg.dataset),\n            os.path.join(script_dir, "data", cfg.dataset),\n            os.path.join("data", cfg.dataset),\n        ]\n        for cand in candidates:\n            if os.path.exists(cand) and os.path.isdir(cand) and os.path.abspath(cand) != os.path.abspath(processed_dir) and len(os.listdir(cand)) > 0:\n                os.makedirs(os.path.dirname(processed_dir), exist_ok=True)\n                try:\n                    os.symlink(cand, processed_dir)\n                    print(f"[DataSet] >>> Auto-bridged symlink: {cand} -> {processed_dir}")\n                except Exception:\n                    import shutil\n                    shutil.copytree(cand, processed_dir, dirs_exist_ok=True)\n                    print(f"[DataSet] >>> Auto-bridged copied: {cand} -> {processed_dir}")\n                break\n\n    # Robust dataset loading:\n    # 1. freerec.data.datasets contains a submodule named \'tiktok\', so getattr(...) returns a module\n    #    rather than a class when cfg.dataset == \'tiktok\'.\n    # 2. FreeRec\'s ValidSampler/TestSampler requires dataset.TASK is MATCHING. We define it on\n    #    RecDataSet class, BaseSet class, and pass tasktag to ensure it is always present.\n    tasktag = getattr(cfg, \'tasktag\', None) or getattr(freerec.data.tags, \'MATCHING\', None)\n    if hasattr(freerec.data.datasets, \'RecDataSet\'):\n        freerec.data.datasets.RecDataSet.TASK = tasktag\n    if hasattr(freerec.data.datasets, \'base\') and hasattr(freerec.data.datasets.base, \'BaseSet\'):\n        freerec.data.datasets.base.BaseSet.TASK = tasktag\n\n    ds_cls = getattr(freerec.data.datasets, cfg.dataset, None)\n    if isinstance(ds_cls, type):\n        try:\n            dataset = ds_cls(root=cfg.root)\n        except Exception:\n            try:\n                from freerec.data.datasets.base import MatchingRecDataSet\n                dataset = MatchingRecDataSet(cfg.root, cfg.dataset, tasktag=tasktag)\n            except Exception:\n                dataset = freerec.data.datasets.RecDataSet(\n                    cfg.root, cfg.dataset, tasktag=tasktag\n                )\n    else:\n        try:\n            from freerec.data.datasets.base import MatchingRecDataSet\n            dataset = MatchingRecDataSet(cfg.root, cfg.dataset, tasktag=tasktag)\n        except Exception:\n            dataset = freerec.data.datasets.RecDataSet(\n                cfg.root, cfg.dataset, tasktag=tasktag\n            )\n\n    # Ensure TASK attribute is always attached to dataset instance\n    if not hasattr(dataset, \'TASK\') or dataset.TASK is None:\n        dataset.TASK = tasktag\n\n    model = STAIR(dataset)\n\n    trainpipe = model.sure_trainpipe(cfg.batch_size)\n    validpipe = model.sure_validpipe(cfg.ranking)\n    testpipe = model.sure_testpipe(cfg.ranking)\n\n    coach = CoachForSTAIR(\n        dataset=dataset,\n        trainpipe=trainpipe,\n        validpipe=validpipe,\n        testpipe=testpipe,\n        model=model,\n        cfg=cfg\n    )\n    coach.fit()\n\n\nif __name__ == "__main__":\n    main()')
with open(os.path.join(STAIR_DIR, 'configs', 'tiktok_MMRec.yaml'), 'w', encoding='utf-8') as f:
    f.write("root: data\ndataset: tiktok\n\nembedding_dim: 64\nnum_layers: 3\n\nepochs: 500\nbatch_size: 1024\noptimizer: adamwsevo\nlr: 1.e-3\nweight_decay: 0.1\n\n# Content-driven micro-video scenario: smaller gamma (0.01 - 0.05) preserves richer multimodal cues in FSC\ngamma: 0.05\n\n# Modality files & kNN neighbor settings (Tri-modal: Vision, Text, Audio)\nmfiles: visual_modality.pkl,textual_modality.pkl,audio_modality.pkl\nnum_neighbors: '3-3-3'\n\nmonitors: [LOSS, Recall@1, Recall@10, Recall@20, NDCG@10, NDCG@20]\nwhich4best: NDCG@20\n")

# 2. Cài đặt các gói phụ thuộc bắt buộc
print("📦 Cài đặt dependencies (torchdata, torch_geometric, freerec, nvidia-ml-py, prettytable, scipy)...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torchdata==0.7.1'], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch_geometric', 'freerec==0.8.5', 'nvidia-ml-py', 'prettytable', 'matplotlib', 'pyyaml', 'seaborn', 'scipy'
], check=True)

# 3. Kaggle TorchData compatibility shims cho FreeRec (PyTorch 2.x & Python 3.10+)
import types
import torch.utils.data

try:
    import torchdata
    import torchdata.datapipes as dp
except Exception:
    dp = None

if dp is None or 'torchdata.datapipes' not in sys.modules:
    if 'torchdata' not in sys.modules:
        td = types.ModuleType('torchdata')
        sys.modules['torchdata'] = td
    else:
        td = sys.modules['torchdata']
    dp = types.ModuleType('torchdata.datapipes')
    td.datapipes = dp
    sys.modules['torchdata.datapipes'] = dp

if not hasattr(dp, 'iter'):
    iter_mod = types.ModuleType('torchdata.datapipes.iter')
    dp.iter = iter_mod
    sys.modules['torchdata.datapipes.iter'] = iter_mod
if not hasattr(dp.iter, 'IterDataPipe'):
    class IterDataPipe(torch.utils.data.IterableDataset):
        def __iter__(self): return iter([])
    dp.iter.IterDataPipe = IterDataPipe

if not hasattr(dp, 'map'):
    map_mod = types.ModuleType('torchdata.datapipes.map')
    dp.map = map_mod
    sys.modules['torchdata.datapipes.map'] = map_mod
if not hasattr(dp.map, 'MapDataPipe'):
    class MapDataPipe(torch.utils.data.Dataset):
        def __getitem__(self, idx): raise NotImplementedError
        def __len__(self): return 0
    dp.map.MapDataPipe = MapDataPipe

if not hasattr(dp, 'functional_datapipe'):
    def functional_datapipe(name, enable_df_datapipes_support=False):
        def decorator(cls):
            def method(self, *args, **kwargs): return cls(self, *args, **kwargs)
            if hasattr(dp, 'iter') and hasattr(dp.iter, 'IterDataPipe'): setattr(dp.iter.IterDataPipe, name, method)
            if hasattr(dp, 'map') and hasattr(dp.map, 'MapDataPipe'): setattr(dp.map.MapDataPipe, name, method)
            return cls
        return decorator
    dp.functional_datapipe = functional_datapipe

print("✅ Môi trường STAIR Baseline & Dependencies đã sẵn sàng 100%!")


## Cell 2 📂 Chuẩn bị Dữ liệu TikTok từ Kaggle Input (Tự động Adapter sang FreeRec)
Tự động quét toàn bộ `/kaggle/input` để phát hiện dữ liệu TikTok (chứa `trnMat.pkl`, `audio_feat.npy`, `image_feat.npy`, `text_feat.npy`, ...).
- Nếu là dữ liệu thô (raw DiffMM / ACM MM 2024), hệ thống tự động chuyển đổi sang chuẩn FreeRec (`train.txt`, `valid.txt`, `test.txt`, `visual_modality.pkl`, `textual_modality.pkl`, `audio_modality.pkl`).
- Tự động liên kết (symlink/bridge) vào `/kaggle/data/Processed/tiktok` và `/kaggle/data/tiktok` sẵn sàng cho huấn luyện.

In [ ]:
# Cell 2: Chuẩn bị và Chuyển đổi Dữ liệu TikTok (Tự động quét & Adapter sang FreeRec)
import os, shutil, glob, time, pickle, zipfile
import numpy as np
import torch
import scipy.sparse as sp

DATA_ROOT = '/kaggle/data'
PROCESSED_ROOT = os.path.join(DATA_ROOT, 'Processed')
LOCAL_DATA = '/kaggle/working/STAIR-Enhanced/data'
LOCAL_PROCESSED = os.path.join(LOCAL_DATA, 'Processed')

for d in [DATA_ROOT, PROCESSED_ROOT, LOCAL_DATA, LOCAL_PROCESSED]:
    os.makedirs(d, exist_ok=True)

TARGET_FOLDER = 'tiktok'
REQUIRED_PROCESSED_FILES = [
    'train.txt', 'valid.txt', 'test.txt',
    'visual_modality.pkl', 'textual_modality.pkl', 'audio_modality.pkl'
]
RAW_FILES = [
    'trnMat.pkl', 'valMat.pkl', 'tstMat.pkl',
    'audio_feat.npy', 'image_feat.npy', 'text_feat.npy'
]

def bridge_directories(src_dir, target_folder):
    destinations = [
        os.path.join(DATA_ROOT, target_folder),
        os.path.join(PROCESSED_ROOT, target_folder),
        os.path.join(LOCAL_DATA, target_folder),
        os.path.join(LOCAL_PROCESSED, target_folder),
    ]
    for dst in destinations:
        if os.path.abspath(src_dir) == os.path.abspath(dst):
            continue
        os.makedirs(dst, exist_ok=True)
        for item in os.listdir(src_dir):
            s_item = os.path.join(src_dir, item)
            d_item = os.path.join(dst, item)
            if os.path.isfile(s_item) and not os.path.exists(d_item):
                try:
                    os.symlink(s_item, d_item)
                except Exception:
                    shutil.copy2(s_item, d_item)

def convert_raw_tiktok(src_dir, dst_dir):
    print(f"🔄 Đang thực hiện chuyển đổi TikTok dataset từ {src_dir} sang chuẩn FreeRec/STAIR ({dst_dir})...")
    os.makedirs(dst_dir, exist_ok=True)
    
    for rf in RAW_FILES:
        src_path = os.path.join(src_dir, rf)
        dst_path = os.path.join(dst_dir, rf)
        if os.path.exists(src_path) and not os.path.exists(dst_path):
            shutil.copy2(src_path, dst_path)

    trn_mat = pickle.load(open(os.path.join(src_dir, "trnMat.pkl"), "rb"))
    val_mat = pickle.load(open(os.path.join(src_dir, "valMat.pkl"), "rb"))
    tst_mat = pickle.load(open(os.path.join(src_dir, "tstMat.pkl"), "rb"))

    num_users, num_items = trn_mat.shape
    print(f"  • Kích thước Ma trận tương tác: {num_users} Users x {num_items} Items")
    print(f"  • Train NNZ: {trn_mat.nnz:,} | Valid NNZ: {val_mat.nnz:,} | Test NNZ: {tst_mat.nnz:,}")

    splits = [
        ("train.txt", trn_mat),
        ("valid.txt", val_mat),
        ("test.txt", tst_mat)
    ]

    for fname, mat in splits:
        out_path = os.path.join(dst_dir, fname)
        if not hasattr(mat, 'row'):
            mat = mat.tocoo()
        rows = mat.row
        cols = mat.col
        order = np.lexsort((cols, rows))
        sorted_rows = rows[order]
        sorted_cols = cols[order]

        with open(out_path, "w", encoding="utf-8") as f:
            f.write("USER\tITEM\tTIMESTAMP\n")
            for u, i in zip(sorted_rows, sorted_cols):
                f.write(f"{u}\t{i}\t0\n")
        print(f"  [Đã sinh] {fname}: {len(sorted_rows):,} tương tác -> {out_path}")

    audio_feat = np.load(os.path.join(src_dir, "audio_feat.npy"))
    image_feat = np.load(os.path.join(src_dir, "image_feat.npy"))
    text_feat  = np.load(os.path.join(src_dir, "text_feat.npy"))

    print(f"  • Multimodal feature shapes:")
    print(f"    - Audio: {audio_feat.shape} ({audio_feat.dtype})")
    print(f"    - Image: {image_feat.shape} ({image_feat.dtype})")
    print(f"    - Text : {text_feat.shape} ({text_feat.dtype})")

    audio_tensor = torch.from_numpy(audio_feat.astype(np.float32))
    image_tensor = torch.from_numpy(image_feat.astype(np.float32))
    text_tensor  = torch.from_numpy(text_feat.astype(np.float32))

    concat_np = np.concatenate([audio_feat.astype(np.float32), image_feat.astype(np.float32), text_feat.astype(np.float32)], axis=1)
    concat_tensor = torch.from_numpy(concat_np)

    modality_maps = {
        "visual_modality.pkl": image_tensor,
        "textual_modality.pkl": text_tensor,
        "audio_modality.pkl": audio_tensor,
        "multimodal_concat.pkl": concat_tensor
    }

    for pkl_name, tensor_data in modality_maps.items():
        out_path = os.path.join(dst_dir, pkl_name)
        with open(out_path, "wb") as f:
            pickle.dump(tensor_data, f, protocol=pickle.HIGHEST_PROTOCOL)
        size_mb = os.path.getsize(out_path) / (1024 * 1024)
        print(f"  [Đã sinh] {pkl_name}: {tensor_data.shape} ({size_mb:.2f} MB)")

    print(f"✅ Hoàn tất chuyển đổi TikTok dataset sang chuẩn STAIR/FreeRec!")

def scan_and_prepare_tiktok():
    target_processed = os.path.join(PROCESSED_ROOT, TARGET_FOLDER)
    
    # 1. Kiểm tra nếu đã có sẵn đầy đủ các tệp processed
    for cand in [target_processed, os.path.join(DATA_ROOT, TARGET_FOLDER), os.path.join(LOCAL_PROCESSED, TARGET_FOLDER)]:
        if os.path.exists(cand) and all(os.path.exists(os.path.join(cand, f)) for f in REQUIRED_PROCESSED_FILES):
            print(f"✅ Dữ liệu TikTok chuẩn FreeRec đã tồn tại sẵn tại: {cand}")
            bridge_directories(cand, TARGET_FOLDER)
            return {'tiktok': target_processed}

    # 2. Tìm kiếm trong /kaggle/input
    print("🔍 Đang quét /kaggle/input để tìm kiếm tập dữ liệu TikTok...")
    input_base = '/kaggle/input'
    raw_src = None
    
    for attempt in range(12):
        if os.path.exists(input_base):
            for root, dirs, files in os.walk(input_base):
                if all(rf in files for rf in REQUIRED_PROCESSED_FILES):
                    print(f"  [TÌM THẤY DỮ LIỆU ĐÃ XỬ LÝ] {root}")
                    os.makedirs(target_processed, exist_ok=True)
                    for f in files:
                        shutil.copy2(os.path.join(root, f), os.path.join(target_processed, f))
                    bridge_directories(target_processed, TARGET_FOLDER)
                    return {'tiktok': target_processed}

                if all(rf in files for rf in RAW_FILES):
                    raw_src = root
                    print(f"  [TÌM THẤY DỮ LIỆU RAW TIKTOK] {root}")
                    break
            if raw_src:
                break

        # Kiểm tra dự phòng từ repo tiktok.zip
        repo_zip = os.path.join(LOCAL_DATA, 'tiktok.zip')
        if not raw_src and os.path.exists(repo_zip):
            print(f"📦 Phát hiện tệp nén {repo_zip} trong repository. Đang giải nén...")
            extract_dir = os.path.join(LOCAL_DATA, 'raw_tiktok')
            with zipfile.ZipFile(repo_zip, 'r') as zf:
                zf.extractall(extract_dir)
            for root, dirs, files in os.walk(extract_dir):
                if all(rf in files for rf in RAW_FILES):
                    raw_src = root
                    break

        if raw_src:
            break
        print(f"⏳ Dataset đang được tải về từ Kaggle... Đợi 5s (lần {attempt+1}/12)...")
        time.sleep(5)

    if raw_src:
        convert_raw_tiktok(raw_src, target_processed)
        bridge_directories(target_processed, TARGET_FOLDER)
        return {'tiktok': target_processed}
    else:
        print("❌ Không tìm thấy tập dữ liệu TikTok trong /kaggle/input hoặc trong repository!")
        return {}

prepared_data = scan_and_prepare_tiktok()
print("=" * 75)
p_dir = os.path.join(PROCESSED_ROOT, TARGET_FOLDER)
ready_files = [f for f in REQUIRED_PROCESSED_FILES if os.path.exists(os.path.join(p_dir, f))]
status = f"✅ {len(ready_files)}/{len(REQUIRED_PROCESSED_FILES)} tệp chuẩn sẵn sàng" if len(ready_files) == len(REQUIRED_PROCESSED_FILES) else "❌ THIẾU"
print(f"TỔNG KẾT DỮ LIỆU TIKTOK: {status} trong {p_dir}")
for rf in REQUIRED_PROCESSED_FILES:
    fp = os.path.join(p_dir, rf)
    if os.path.exists(fp):
        sz = os.path.getsize(fp) / (1024 * 1024)
        print(f"  • {rf:25s}: {sz:.2f} MB")
print("=" * 75)


## Cell 3 🛠️ Runner Engine: Huấn luyện STAIR Baseline & Trích xuất 4 Chỉ số Khoa học
Xây dựng hàm thực thi huấn luyện STAIR Baseline nguyên bản qua `main.py`:
- Giám sát bộ nhớ VRAM định kỳ (`pynvml`).
- Luồng output trực tiếp thời gian thực ra màn hình và lưu toàn bộ log ra `/kaggle/working/logs/tiktok_baseline.log`.
- Tự động trích xuất các chỉ số tại Checkpoint tốt nhất: **Recall@10, Recall@20, NDCG@10, NDCG@20**.

In [ ]:
# Cell 3: Runner Engine cho STAIR Baseline
import subprocess, sys, os, time, re, threading

try:
    import pynvml
    pynvml.nvmlInit()
    HAS_NVML = True
except Exception:
    HAS_NVML = False

def get_gpu_memory_used():
    if not HAS_NVML:
        return 0.0
    try:
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        return info.used / (1024 ** 2)
    except Exception:
        return 0.0

TRACKED_METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

def extract_baseline_metrics(log_path):
    if not os.path.exists(log_path):
        return None, {}

    best_epoch = None
    best_metrics = {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    # 1. Tìm best epoch từ dòng 'Load best model @Epoch'
    for line in lines:
        m = re.search(r'Load best model @Epoch\s+(\d+)', line)
        if m:
            best_epoch = int(m.group(1))

    # 2. Tìm phần TEST tại best epoch ở cuối log
    for line in reversed(lines):
        if 'TEST' in line and any(k in line for k in ['NDCG', 'RECALL']):
            for metric in TRACKED_METRICS:
                pattern = rf'{re.escape(metric)}[\sAvg:]+([0-9.]+)'
                m = re.search(pattern, line, re.IGNORECASE)
                if m and metric not in best_metrics:
                    best_metrics[metric] = float(m.group(1))
            if len(best_metrics) >= len(TRACKED_METRICS):
                break

    return best_epoch, best_metrics

def run_stair_baseline(yaml_cfg, data_root, log_path):
    print('=' * 85)
    print('🚀 BẮT ĐẦU HUẤN LUYỆN: STAIR BASELINE (AAAI 2025) TRÊN TẬP TIKTOK')
    print(f'  * Model Architecture : STAIR Baseline (Original)')
    print(f'  * Config File        : {yaml_cfg}')
    print(f'  * Data Root          : {data_root}')
    print(f'  * Log Path           : {log_path}')
    print('=' * 85)

    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    t0 = time.time()
    runner_py = '/kaggle/working/STAIR-Enhanced/main.py'
    if not os.path.exists(runner_py):
        runner_py = 'main.py'

    cmd = [
        sys.executable, runner_py,
        '--config', yaml_cfg,
        '--root',   data_root,
    ]

    with open(log_path, 'w', encoding='utf-8') as f:
        proc = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, universal_newlines=True
        )
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            f.write(line)
            f.flush()
        proc.wait()

    elapsed = time.time() - t0
    print('=' * 85)
    if proc.returncode != 0:
        print(f'❌ [THẤT BẠI] Quá trình huấn luyện STAIR Baseline gặp lỗi (Exit Code: {proc.returncode})!')
    else:
        print(f'✅ [HOÀN TẤT] Huấn luyện STAIR Baseline hoàn thành trong {elapsed/60:.2f} phút ({elapsed:.1f}s)!')

    best_ep, metrics = extract_baseline_metrics(log_path)
    print(f'  * Checkpoint tối ưu : Epoch {best_ep}')
    for m, val in metrics.items():
        print(f'  * {m:12s}: {val:.4f}')
    print('=' * 85)


## Cell 4 📋 Cấu hình Siêu tham số STAIR Baseline cho TikTok
Tệp cấu hình chính thức `configs/tiktok_MMRec.yaml`:
- `embedding_dim`: 64 (Chuẩn baseline paper STAIR AAAI 2025).
- `batch_size`: 1024 | `epochs`: 500 | `eval_freq`: 5 eps.
- `gamma`: 0.05 (Hệ số suy giảm phổ FSC cho video ngắn).
- `mfiles`: visual, textual, audio | `num_neighbors`: 3-3-3.
- `optimizer`: adamwsevo ($lr = 10^{-3}$, weight decay $= 0.1$).

In [ ]:
# Cell 4: Xác nhận cấu hình STAIR Baseline TikTok
import yaml

yaml_path = '/kaggle/working/STAIR-Enhanced/configs/tiktok_MMRec.yaml'
with open(yaml_path, 'r', encoding='utf-8') as f:
    cfg_dict = yaml.safe_load(f)

print('=' * 75)
print('THIẾT LẬP THAM SỐ CHUẨN TẮC CHO STAIR BASELINE TIKTOK:')
for k, v in cfg_dict.items():
    print(f'  • {k:18s}: {v}')
print('=' * 75)


## Cell 5 🏋️ Thực thi Huấn luyện STAIR Baseline trên TikTok
Khởi chạy huấn luyện mô hình STAIR Baseline nguyên bản.
Tiến trình huấn luyện sẽ hiển thị trực tiếp và lưu vào `/kaggle/working/logs/tiktok_baseline.log`.

In [ ]:
# Cell 5: Huấn luyện STAIR Baseline trên TikTok
DATA_ROOT = '/kaggle/data'
YAML_CFG = '/kaggle/working/STAIR-Enhanced/configs/tiktok_MMRec.yaml'
LOG_PATH = '/kaggle/working/logs/tiktok_baseline.log'

if 'tiktok' in prepared_data or (os.path.exists('/kaggle/data/Processed/tiktok') and len(os.listdir('/kaggle/data/Processed/tiktok')) >= 6):
    run_stair_baseline(
        yaml_cfg=YAML_CFG,
        data_root=DATA_ROOT,
        log_path=LOG_PATH,
    )
else:
    print('⚠️ Bỏ qua huấn luyện do thiếu dữ liệu TikTok trong /kaggle/input.')


## Cell 6 📊 Bảng Đối Soát Kết Quả với Log Mốc Ngày 14/09/2026
Bảng tổng hợp đối sánh giữa kết quả vừa chạy và mốc chuẩn trong `logs/paper/tiktok.log` (Epoch 220) trên cả 4 chỉ số khoa học: **Recall@10, Recall@20, NDCG@10, NDCG@20**.

In [ ]:
# Cell 6: Bảng đối soát kết quả STAIR Baseline TikTok
from prettytable import PrettyTable

# Mốc chuẩn chính thức trong logs/paper/tiktok.log (Best @Epoch 220)
HISTORICAL_BASELINE = {
    'epoch': 220,
    'Recall@10': 0.0558,
    'Recall@20': 0.0799,
    'NDCG@10':   0.0292,
    'NDCG@20':   0.0352,
}

log_file = '/kaggle/working/logs/tiktok_baseline.log'
best_ep, current_metrics = extract_baseline_metrics(log_file)

table = PrettyTable()
table.field_names = ['Lần chạy', 'Checkpoint', 'Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20', 'Trạng thái']

hb = HISTORICAL_BASELINE
table.add_row([
    'Log Mốc (14/09)', f'Epoch {hb["epoch"]}',
    f'{hb["Recall@10"]:.4f}', f'{hb["Recall@20"]:.4f}',
    f'{hb["NDCG@10"]:.4f}', f'{hb["NDCG@20"]:.4f}',
    'Mốc chuẩn paper'
])

if current_metrics and len(current_metrics) >= 4:
    cm = current_metrics
    diff_n20 = cm['NDCG@20'] - hb['NDCG@20']
    status_str = f'Khớp chuẩn (Δ={diff_n20:+.4f})' if abs(diff_n20) < 0.002 else f'Chênh lệch (Δ={diff_n20:+.4f})'
    table.add_row([
        'Tái lập hiện tại', f'Epoch {best_ep}',
        f'{cm["Recall@10"]:.4f}', f'{cm["Recall@20"]:.4f}',
        f'{cm["NDCG@10"]:.4f}', f'{cm["NDCG@20"]:.4f}',
        status_str
    ])
else:
    table.add_row(['Tái lập hiện tại', 'Chưa hoàn tất', 'N/A', 'N/A', 'N/A', 'N/A', 'Đang chờ log'])

print('=' * 85)
print('BẢNG TỔNG KẾT ĐỐI SOÁT STAIR BASELINE TRÊN TẬP TIKTOK:')
print(table)
print('=' * 85)
